In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_4")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

condition_cycle_map = {}
for condition in condition_pal_map:
    for cycle in range(10, 15):
        condition_cycle_map[f"{condition}-{cycle}"] = condition_pal_map[condition][cycle-10]

all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

In [ ]:
from collections import defaultdict

fig, axes = plt.subplots(4, 1, figsize=(6, 5), sharex=True, sharey=True)

from matplotlib.colors import LinearSegmentedColormap

custom_csv = pd.read_csv(r"C:\Tracking\BlastodermAnalysis\figures\blender\ccc-tool_colormap_Custom CMS4.csv", delimiter=";", skiprows=0)

cmap = LinearSegmentedColormap.from_list("custom_cmap", custom_csv[["R", "G", "B"]].values, N=256)

condition_cycle_speeds = defaultdict(list)

for cycle, ax in zip([11, 12, 13, 14], axes):

    for k, df in enumerate(spots_dfs[:3]):
        condition = "wt"
        if condition_map[stems[k][:-6]] != condition:
            continue
        df = df.query("AP.between(0.2, 0.8)")

        time_window = (-6, 10)

        track_df = df.groupby(["track_id", "frame"]).agg({
            "AP": "mean",
            "pseudotime": "mean",
            "dAP": "mean",
            "cycle": "mean",
            "distance": "mean",
            "time_since_nc11": "mean",
            "tracklet_id": "nunique",
            "x": "mean",
            "y": "mean",
            "z": "mean",
        }).reset_index()

        for col in ["x", "y", "z", "time_since_nc11"]:
            track_df[col] = track_df.groupby("track_id")[col].transform(lambda x: x.rolling(5, center=True, min_periods=1).mean())
            track_df[f"d_{col}"] = track_df.groupby("track_id")[col].diff()

        track_df["speed"] = np.sqrt(track_df["d_x"]**2 + track_df["d_y"]**2 + track_df["d_z"]**2) / track_df["d_time_since_nc11"]

        t = track_df.groupby("track_id")["tracklet_id"].max()
        t = t[t > 14]
        track_df = track_df[track_df["track_id"].isin(t.index)].copy()

        track_first_cycle_time = track_df.query("cycle > @cycle - 0.6 and cycle < @cycle + 0.1").groupby("track_id")["time_since_nc11"].min()

        track_df["time_since_cycle"] = track_df["time_since_nc11"] - track_df["track_id"].map(track_first_cycle_time)
        track_df["time_since_cycle"] = (track_df["time_since_cycle"] // 0.33) * 0.33

        windowed = track_df.query("time_since_cycle > @time_window[0] and time_since_cycle < @time_window[1]")

        print(k, windowed["speed"].mean())

        condition_cycle_speeds["src"].append(k)
        condition_cycle_speeds["condition"].append(condition)
        condition_cycle_speeds["cycle"].append(cycle)
        condition_cycle_speeds["speed"].append(windowed["speed"].mean())

        sns.lineplot(windowed, x="time_since_cycle", y="speed", legend=False, ax=ax, color=cmap((cycle - 12)*0.1 + 0.5), lw=3, errorbar=None)

    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_ylim(0, 8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.axvline(0, color="k", ls="--")

fig.supylabel("Speed (µm/min)")
fig.supxlabel("Time since mitosis")
plt.savefig(save_path / f"nuclear_speed_around_mitosis.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
from collections import defaultdict

fig, axes = plt.subplots(4, 1, figsize=(6, 5), sharex=True, sharey=True)

from matplotlib.colors import LinearSegmentedColormap

custom_csv = pd.read_csv(r"C:\Tracking\BlastodermAnalysis\figures\blender\ccc-tool_colormap_Custom CMS4.csv", delimiter=";", skiprows=0)

cmap = LinearSegmentedColormap.from_list("custom_cmap", custom_csv[["R", "G", "B"]].values, N=256)

condition_cycle_speeds = defaultdict(list)

for cycle, ax in zip([11, 12, 13, 14], axes):

    for k, df in enumerate(spots_dfs):
        if k == 3 or k == 10:
            continue

        condition = condition_map[stems[k][:-6]]
        if condition == "bcd":
            continue

        df = df.query("AP.between(0.2, 0.8)")

        time_window = (-6, 10)

        track_df = df.groupby(["track_id", "frame"]).agg({
            "AP": "mean",
            "pseudotime": "mean",
            "dAP": "mean",
            "cycle": "mean",
            "distance": "mean",
            "time_since_nc11": "mean",
            "tracklet_id": "nunique",
            "x": "mean",
            "y": "mean",
            "z": "mean",
        }).reset_index()

        for col in ["x", "y", "z", "time_since_nc11"]:
            track_df[col] = track_df.groupby("track_id")[col].transform(lambda x: x.rolling(5, center=True, min_periods=1).mean())
            track_df[f"d_{col}"] = track_df.groupby("track_id")[col].diff()

        track_df["speed"] = np.sqrt(track_df["d_x"]**2 + track_df["d_y"]**2 + track_df["d_z"]**2) / track_df["d_time_since_nc11"]

        t = track_df.groupby("track_id")["tracklet_id"].max()
        t = t[t > 14]
        track_df = track_df[track_df["track_id"].isin(t.index)].copy()

        track_first_cycle_time = track_df.query("cycle > @cycle - 0.6 and cycle < @cycle + 0.1").groupby("track_id")["time_since_nc11"].min()

        track_df["time_since_cycle"] = track_df["time_since_nc11"] - track_df["track_id"].map(track_first_cycle_time)
        track_df["time_since_cycle"] = (track_df["time_since_cycle"] // 0.33) * 0.33

        windowed = track_df.query("time_since_cycle > @time_window[0] and time_since_cycle < @time_window[1]")

        print(k, windowed["speed"].mean())

        condition_cycle_speeds["src"].append(k)
        condition_cycle_speeds["condition"].append(condition)
        condition_cycle_speeds["cycle"].append(cycle)
        condition_cycle_speeds["speed"].append(windowed["speed"].mean())

        sns.lineplot(windowed, x="time_since_cycle", y="speed", legend=False, ax=ax, color=condition_pal_map[condition][2], lw=2, errorbar=None)

    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_ylim(0, 8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.axvline(0, color="k", ls="--")

fig.supylabel("Speed (µm/min)")
fig.supxlabel("Time since mitosis")
plt.savefig(save_path / f"nuclear_speed_around_mitosis.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
region_colors = np.array(["#0a9396", "#ee9b00", "#ae2012"])
regions = ["Anterior", "Middle", "Posterior"]

k = 0
df = spots_dfs[k]

t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "cycle"]].mean().reset_index()

t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())
t["AP_from_start_percent"] = t["AP_from_start"] * 100
t["time_since_nc11"] = np.round(t["time_since_nc11"], 3)

fig, axes = plt.subplots(figsize=(4.5, 2.2))

for i, ap_group in enumerate([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]):

    region_track_ids = t[t["AP"].between(*ap_group)]["track_id"].unique()
    early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
    early_track_ids = t["track_id"].unique()[early_track_ids]

    all_good = np.intersect1d(region_track_ids, early_track_ids)
    t_new = t[t["track_id"].isin(all_good)].copy()

    sns.lineplot(t_new, x="time_since_nc11", y="AP_from_start_percent", color=region_colors[i], errorbar=None, alpha=1, label=regions[i], linewidth=4, legend=False)

    for cycle in cycles[1:]:
        cycle_df = t[t["cycle"] == cycle]
        cycle_times = cycle_df.groupby("track_id")["time_since_nc11"].min()
        division_time = cycle_times.median()
        plt.axvline(division_time, color="k", linestyle="--", linewidth=2, alpha=0.2)

    axes.spines["top"].set_visible(False)
    axes.spines["right"].set_visible(False)

axes.invert_yaxis()
plt.title(f"Average displacement of region over time")
plt.ylabel("Ap displacement (% Embryo length)")
plt.xlabel("Time since nc11 (minutes)")
plt.ylim(-6, 10)
# plt.legend()
plt.savefig(save_path / f"{stems[k]}_nuclear_movement_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
region_colors = np.array(["#0a9396", "#ee9b00", "#ae2012"])
regions = ["Anterior", "Middle", "Posterior"]

k = 7
df = spots_dfs[k]

t = df.groupby(["track_id", "frame"])[["x", "y", "z", "time_since_nc11", "AP", "theta", "cycle"]].mean().reset_index()

t["AP_from_start"] = t["AP"] - t["track_id"].map(t.groupby("track_id")["AP"].first())
t["AP_from_start_percent"] = t["AP_from_start"] * 100
t["time_since_nc11"] = np.round(t["time_since_nc11"], 3)

fig, axes = plt.subplots(figsize=(4.5, 2.2))

for i, ap_group in enumerate([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)]):

    region_track_ids = t[t["AP"].between(*ap_group)]["track_id"].unique()
    early_track_ids = t.groupby("track_id")["time_since_nc11"].min() < 0
    early_track_ids = t["track_id"].unique()[early_track_ids]

    all_good = np.intersect1d(region_track_ids, early_track_ids)
    t_new = t[t["track_id"].isin(all_good)].copy()

    sns.lineplot(t_new, x="time_since_nc11", y="AP_from_start_percent", color=region_colors[i], errorbar=None, alpha=1, label=regions[i], linewidth=4, legend=False)

    for cycle in cycles[1:]:
        cycle_df = t[t["cycle"] == cycle]
        cycle_times = cycle_df.groupby("track_id")["time_since_nc11"].min()
        division_time = cycle_times.median()
        plt.axvline(division_time, color="k", linestyle="--", linewidth=2, alpha=0.2)

    axes.spines["top"].set_visible(False)
    axes.spines["right"].set_visible(False)

axes.invert_yaxis()
plt.title(f"Average displacement of region over time")
plt.ylabel("Ap displacement (% Embryo length)")
plt.xlabel("Time since nc11 (minutes)")
plt.ylim(-6, 10)
# plt.legend()
plt.savefig(save_path / f"{stems[k]}_nuclear_movement_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print(spots_dfs[0].columns)

In [ ]:
for k, df in enumerate(spots_dfs[:3]):
    sns.lineplot(df, x="frame", y="dtot", color="w", errorbar=None)

    frames = all_mmfs[stems[k]]
    for frame in frames:
        plt.axvline(frame, color="r", ls="--")
    plt.show()

In [ ]:
# displacement since previous nuclear cycle
displacements_df = defaultdict(list)

for cycle in [11, 12, 13, 14]:

    for k, df in enumerate(spots_dfs[:]):
        condition = "wt"
        if condition_map[stems[k][:-6]] != condition:
            continue

        condition = condition_map[stems[k][:-6]]
        df = df.query("AP.between(0.75, 0.95)")

        track_df = df.groupby(["track_id", "frame"]).agg({
            "AP": "mean",
            "pseudotime": "mean",
            "dAP": "mean",
            "cycle": "mean",
            "distance": "mean",
            "time_since_nc11": "mean",
            "tracklet_id": "nunique",
            "x": "mean",
            "y": "mean",
            "z": "mean",
        }).reset_index()

        current_cycle_frame = all_mmfs[stems[k]][cycle-10]
        prev_cycle_frame = all_mmfs[stems[k]][cycle-11]

        current_frame_df = track_df[track_df["frame"] == current_cycle_frame][["track_id", "x", "y", "z"]].copy()
        prev_frame_df = track_df[track_df["frame"] == prev_cycle_frame][["track_id", "x", "y", "z"]].copy()

        for col in ["x", "y", "z"]:
            current_frame_df[f"d_{col}"] = current_frame_df[col] - current_frame_df["track_id"].map(prev_frame_df.set_index("track_id")[col])


        current_frame_df["displacement"] = np.sqrt(current_frame_df["d_x"]**2 + current_frame_df["d_y"]**2 + current_frame_df["d_z"]**2)

        displacements_df["src"].append(k)
        displacements_df["condition"].append(condition)
        displacements_df["cycle"].append(cycle)
        displacements_df["displacement"].append(current_frame_df["displacement"].mean())

sns.barplot(
    data=pd.DataFrame(displacements_df), x="cycle", y="displacement", hue="condition",
    palette=condition_main_colors, errorbar=None)
sns.stripplot(
    data=pd.DataFrame(displacements_df), x="cycle", y="displacement", hue="condition",
    palette="dark:k", dodge=True, alpha=0.5, size=5)

plt.show()

In [ ]:
# displacement since previous nuclear cycle
displacements_df = defaultdict(list)

for cycle in [11, 12, 13, 14]:

    for k, df in enumerate(spots_dfs[:3]):
        condition = "wt"
        if condition_map[stems[k][:-6]] != condition:
            continue

        condition = condition_map[stems[k][:-6]]
        for region, region_name in zip([(0.05, 0.25), (0.4, 0.6), (0.75, 0.95)], ["Anterior", "Middle", "Posterior"]):
            region_df = df.query("AP.between(@region[0], @region[1])")

            track_df = region_df.groupby(["track_id", "frame"]).agg({
                "AP": "mean",
                "pseudotime": "mean",
                "dAP": "mean",
                "cycle": "mean",
                "distance": "mean",
                "time_since_nc11": "mean",
                "tracklet_id": "nunique",
                "x": "mean",
                "y": "mean",
                "z": "mean",
            }).reset_index()

            current_cycle_frame = all_mmfs[stems[k]][cycle-10]
            prev_cycle_frame = all_mmfs[stems[k]][cycle-11]

            if cycle == 11:
                prev_cycle_frame = df["frame"].min()

            current_frame_df = track_df[track_df["frame"] == current_cycle_frame][["track_id", "x", "y", "z"]].copy()
            prev_frame_df = track_df[track_df["frame"] == prev_cycle_frame][["track_id", "x", "y", "z"]].copy()

            for col in ["x", "y", "z"]:
                current_frame_df[f"d_{col}"] = current_frame_df[col] - current_frame_df["track_id"].map(prev_frame_df.set_index("track_id")[col])


            current_frame_df["displacement"] = np.sqrt(current_frame_df["d_x"]**2 + current_frame_df["d_y"]**2 + current_frame_df["d_z"]**2)

            displacements_df["src"].append(k)
            displacements_df["condition"].append(condition)
            displacements_df["cycle"].append(cycle)
            displacements_df["displacement"].append(current_frame_df["displacement"].mean())
            displacements_df["region"].append(region_name)

sns.barplot(
    data=pd.DataFrame(displacements_df), x="cycle", y="displacement", hue="region", palette=list(region_colors), errorbar=None)
sns.stripplot(
    data=pd.DataFrame(displacements_df), x="cycle", y="displacement", hue="region", palette="dark:k", dodge=True, alpha=0.5, size=5, legend=False)

plt.ylabel("Displacement (µm)")
plt.xlabel("Cycle")
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.legend(title="Region", loc="upper right", bbox_to_anchor=(1, 1))
plt.savefig(save_path / f"displacement_since_previous_cycle_by_region.png", dpi=300, bbox_inches="tight")
plt.show()